# Model Comparison Report — Student Attention Detection

## TUBITAK 2209-B Projesi

Bu notebook, egitilen 4 farkli duygu analizi modelini karsilastirir ve en iyi modeli belirler.

> **Not:** Bu notebook `02_model_training.ipynb` ile egitilen modellerin sonuclarini kullanir. Oncelikle egitim notebook'unu calistirdiginizdan emin olun.

### Karsilastirilan Modeller
| Model | Mimari | Ozellik |
|-------|--------|---------|
| EfficientNet-B3 | EfficientNet | Ana model, yuksek kapasite |
| EfficientNet-B0 | EfficientNet | Hafif versiyon |
| MobileNetV3-Large | MobileNet | Mobil optimize |
| ResNet50+CBAM | ResNet + Attention | CBAM dikkat modulu |

### Degerlendirme Kriterleri
- F1-macro (%40) — sinif bazli dengeli performans
- Inference suresi (%25) — gercek zamanli kullanim
- Accuracy (%20) — genel dogruluk
- Model boyutu (%15) — deploy kolayligi

In [ ]:
# ============================================================================
# Cell 2: Setup — Mount Drive, imports, paths
# ============================================================================

from google.colab import drive
drive.mount('/content/drive')

import json
import os
import glob

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, HTML

# Paths on Google Drive
DRIVE_BASE = '/content/drive/MyDrive/Tubitak-2209B'
RESULTS_DIR = f'{DRIVE_BASE}/results'
CHECKPOINTS_DIR = f'{DRIVE_BASE}/checkpoints'
ONNX_DIR = f'{DRIVE_BASE}/onnx_models'

# Model names
MODEL_NAMES = ['efficientnet_b3', 'efficientnet_b0', 'mobilenet_v3', 'resnet50_cbam']
DISPLAY_NAMES = {
    'efficientnet_b3': 'EfficientNet-B3',
    'efficientnet_b0': 'EfficientNet-B0',
    'mobilenet_v3': 'MobileNetV3-Large',
    'resnet50_cbam': 'ResNet50+CBAM',
}
CLASS_NAMES = ['negative', 'neutral', 'positive']

# Plot style
sns.set_theme(style='whitegrid', font_scale=1.1)
MODEL_COLORS = {
    'efficientnet_b3': '#2ecc71',
    'efficientnet_b0': '#3498db',
    'mobilenet_v3': '#e74c3c',
    'resnet50_cbam': '#f39c12',
}

print(f'Results dir: {RESULTS_DIR}')
print(f'Checkpoints dir: {CHECKPOINTS_DIR}')
print(f'ONNX dir: {ONNX_DIR}')
print(f'Models to compare: {MODEL_NAMES}')

In [ ]:
# ============================================================================
# Cell 3: Load results — all {model}_metrics.json files
# ============================================================================

all_metrics = []
loaded_models = []
missing_models = []

for model_name in MODEL_NAMES:
    metrics_path = os.path.join(RESULTS_DIR, f'{model_name}_metrics.json')
    if os.path.exists(metrics_path):
        with open(metrics_path, 'r') as f:
            metrics = json.load(f)
        metrics['model_key'] = model_name
        all_metrics.append(metrics)
        loaded_models.append(model_name)
        print(f'[OK] Loaded: {metrics_path}')
    else:
        missing_models.append(model_name)
        print(f'[MISSING] Not found: {metrics_path}')

if missing_models:
    print(f'\nWARNING: {len(missing_models)} model(s) missing: {missing_models}')
    print('Comparison will proceed with available models only.')

if not all_metrics:
    raise FileNotFoundError(
        f'No metrics files found in {RESULTS_DIR}. '
        'Please run training notebook first.'
    )

# Build DataFrame
rows = []
for m in all_metrics:
    rows.append({
        'model_key': m['model_key'],
        'Model': DISPLAY_NAMES[m['model_key']],
        'Params (M)': m.get('params', 0) / 1e6,
        'Accuracy': m.get('accuracy', 0),
        'F1 (Macro)': m.get('f1_macro', 0),
        'F1 (Weighted)': m.get('f1_weighted', 0),
        'Inference (ms)': m.get('inference_ms', 0),
    })

df = pd.DataFrame(rows)
print(f'\nLoaded {len(df)} model results.')
df.head()

In [ ]:
# ============================================================================
# Cell 4: Display comparison table — styled DataFrame
# ============================================================================

display_df = df[['Model', 'Params (M)', 'Accuracy', 'F1 (Macro)', 'F1 (Weighted)', 'Inference (ms)']].copy()
display_df = display_df.sort_values('Accuracy', ascending=False).reset_index(drop=True)

styled = (
    display_df.style
    .format({
        'Params (M)': '{:.2f}',
        'Accuracy': '{:.4f}',
        'F1 (Macro)': '{:.4f}',
        'F1 (Weighted)': '{:.4f}',
        'Inference (ms)': '{:.2f}',
    })
    .highlight_max(subset=['Accuracy', 'F1 (Macro)', 'F1 (Weighted)'], color='#a8e6cf')
    .highlight_min(subset=['Inference (ms)', 'Params (M)'], color='#ffd3b6')
    .set_caption('Model Comparison — Student Attention Detection')
    .set_table_styles([
        {'selector': 'caption', 'props': [('font-size', '16px'), ('font-weight', 'bold')]},
        {'selector': 'th', 'props': [('background-color', '#f0f0f0')]},
    ])
)

display(styled)

# Summary stats
best_acc_row = display_df.loc[display_df['Accuracy'].idxmax()]
best_f1_row = display_df.loc[display_df['F1 (Macro)'].idxmax()]
fastest_row = display_df.loc[display_df['Inference (ms)'].idxmin()]
smallest_row = display_df.loc[display_df['Params (M)'].idxmin()]

print(f"\nHighlights:")
print(f"  Best accuracy:      {best_acc_row['Model']} ({best_acc_row['Accuracy']:.4f})")
print(f"  Best F1 (macro):    {best_f1_row['Model']} ({best_f1_row['F1 (Macro)']:.4f})")
print(f"  Fastest inference:  {fastest_row['Model']} ({fastest_row['Inference (ms)']:.2f} ms)")
print(f"  Smallest model:     {smallest_row['Model']} ({smallest_row['Params (M)']:.2f}M params)")

In [ ]:
# ============================================================================
# Cell 5: Bar charts — Accuracy, F1-Macro, Inference Time
# ============================================================================

plot_df = df.sort_values('Accuracy', ascending=False).reset_index(drop=True)
colors = [MODEL_COLORS[k] for k in plot_df['model_key']]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Test Accuracy
bars = axes[0].barh(plot_df['Model'], plot_df['Accuracy'], color=colors, edgecolor='white')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Test Accuracy')
axes[0].set_xlim(0, 1)
for bar, val in zip(bars, plot_df['Accuracy']):
    axes[0].text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontweight='bold', fontsize=10)

# F1 (Macro)
bars = axes[1].barh(plot_df['Model'], plot_df['F1 (Macro)'], color=colors, edgecolor='white')
axes[1].set_xlabel('F1 Score')
axes[1].set_title('F1 (Macro)')
axes[1].set_xlim(0, 1)
for bar, val in zip(bars, plot_df['F1 (Macro)']):
    axes[1].text(val + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:.4f}', va='center', fontweight='bold', fontsize=10)

# Inference Time
bars = axes[2].barh(plot_df['Model'], plot_df['Inference (ms)'], color=colors, edgecolor='white')
axes[2].set_xlabel('ms / frame')
axes[2].set_title('Inference Time')
for bar, val in zip(bars, plot_df['Inference (ms)']):
    axes[2].text(val + 0.2, bar.get_y() + bar.get_height() / 2,
                 f'{val:.2f}', va='center', fontweight='bold', fontsize=10)

plt.suptitle('Model Comparison — Key Metrics', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'comparison_bar_charts.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Bar charts saved to results/')

In [ ]:
# ============================================================================
# Cell 6: Weighted scoring — pick the best model
# ============================================================================

# Weights for each criterion
WEIGHTS = {
    'f1_macro': 0.40,
    'inference_time': 0.25,
    'accuracy': 0.20,
    'model_size': 0.15,
}

score_df = df[['model_key', 'Model', 'Params (M)', 'Accuracy', 'F1 (Macro)', 'Inference (ms)']].copy()

# Normalize each metric to [0, 1]
# Higher is better: accuracy, f1_macro
for col in ['Accuracy', 'F1 (Macro)']:
    col_min = score_df[col].min()
    col_max = score_df[col].max()
    if col_max - col_min > 0:
        score_df[f'{col}_norm'] = (score_df[col] - col_min) / (col_max - col_min)
    else:
        score_df[f'{col}_norm'] = 1.0

# Lower is better: inference_ms, params — invert so higher = better
for col in ['Inference (ms)', 'Params (M)']:
    col_min = score_df[col].min()
    col_max = score_df[col].max()
    if col_max - col_min > 0:
        score_df[f'{col}_norm'] = 1.0 - (score_df[col] - col_min) / (col_max - col_min)
    else:
        score_df[f'{col}_norm'] = 1.0

# Compute weighted score
score_df['Weighted Score'] = (
    WEIGHTS['f1_macro'] * score_df['F1 (Macro)_norm'] +
    WEIGHTS['inference_time'] * score_df['Inference (ms)_norm'] +
    WEIGHTS['accuracy'] * score_df['Accuracy_norm'] +
    WEIGHTS['model_size'] * score_df['Params (M)_norm']
)

# Ranking
ranking = score_df[['Model', 'Accuracy', 'F1 (Macro)', 'Inference (ms)', 'Params (M)', 'Weighted Score']].copy()
ranking = ranking.sort_values('Weighted Score', ascending=False).reset_index(drop=True)
ranking.index = ranking.index + 1  # 1-based ranking
ranking.index.name = 'Rank'

print('=' * 70)
print('WEIGHTED MODEL RANKING')
print(f'Weights: F1-macro={WEIGHTS["f1_macro"]:.0%}, '
      f'Inference={WEIGHTS["inference_time"]:.0%}, '
      f'Accuracy={WEIGHTS["accuracy"]:.0%}, '
      f'Size={WEIGHTS["model_size"]:.0%}')
print('=' * 70)
display(ranking.style.format({
    'Accuracy': '{:.4f}',
    'F1 (Macro)': '{:.4f}',
    'Inference (ms)': '{:.2f}',
    'Params (M)': '{:.2f}',
    'Weighted Score': '{:.4f}',
}).bar(subset=['Weighted Score'], color='#a8e6cf'))

best_model_name = ranking.iloc[0]['Model']
best_score = ranking.iloc[0]['Weighted Score']
print(f'\n>>> BEST MODEL: {best_model_name} (Score: {best_score:.4f}) <<<')

# Normalized scores breakdown
print('\nNormalized scores per criterion:')
norm_cols = ['Model', 'F1 (Macro)_norm', 'Inference (ms)_norm', 'Accuracy_norm', 'Params (M)_norm']
norm_display = score_df[norm_cols].copy()
norm_display.columns = ['Model', 'F1-Macro (40%)', 'Inference (25%)', 'Accuracy (20%)', 'Size (15%)']
display(norm_display.style.format({c: '{:.3f}' for c in norm_display.columns if c != 'Model'}))

In [ ]:
# ============================================================================
# Cell 7: Side-by-side confusion matrices
# ============================================================================

cm_paths = {}
for model_name in loaded_models:
    path = os.path.join(RESULTS_DIR, f'{model_name}_confusion_matrix.png')
    if os.path.exists(path):
        cm_paths[model_name] = path
    else:
        print(f'[MISSING] Confusion matrix not found: {path}')

if cm_paths:
    n = len(cm_paths)
    cols = min(n, 2)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
    if n == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()

    for idx, (model_name, path) in enumerate(cm_paths.items()):
        img = mpimg.imread(path)
        axes[idx].imshow(img)
        axes[idx].set_title(DISPLAY_NAMES[model_name], fontsize=13, fontweight='bold')
        axes[idx].axis('off')

    # Hide unused axes
    for idx in range(n, len(axes)):
        axes[idx].axis('off')

    plt.suptitle('Confusion Matrices — All Models', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'comparison_confusion_matrices.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Confusion matrices grid saved.')
else:
    print('No confusion matrix images found. Skipping.')

In [ ]:
# ============================================================================
# Cell 8: Side-by-side ROC curves
# ============================================================================

roc_paths = {}
for model_name in loaded_models:
    path = os.path.join(RESULTS_DIR, f'{model_name}_roc.png')
    if os.path.exists(path):
        roc_paths[model_name] = path
    else:
        print(f'[MISSING] ROC curve not found: {path}')

if roc_paths:
    n = len(roc_paths)
    cols = min(n, 2)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(7 * cols, 6 * rows))
    if n == 1:
        axes = np.array([axes])
    axes = np.array(axes).flatten()

    for idx, (model_name, path) in enumerate(roc_paths.items()):
        img = mpimg.imread(path)
        axes[idx].imshow(img)
        axes[idx].set_title(DISPLAY_NAMES[model_name], fontsize=13, fontweight='bold')
        axes[idx].axis('off')

    for idx in range(n, len(axes)):
        axes[idx].axis('off')

    plt.suptitle('ROC Curves — All Models', fontsize=15, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'comparison_roc_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('ROC curves grid saved.')
else:
    print('No ROC curve images found. Skipping.')

In [ ]:
# ============================================================================
# Cell 9: Training curves comparison — val_accuracy on same plot
# ============================================================================

import torch

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
has_history = False

for model_name in loaded_models:
    # Try loading training history from checkpoint
    ckpt_path = os.path.join(CHECKPOINTS_DIR, f'{model_name}_best.pth')
    history = None

    if os.path.exists(ckpt_path):
        try:
            ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
            if 'history' in ckpt:
                history = ckpt['history']
        except Exception as e:
            print(f'[WARN] Could not load checkpoint for {model_name}: {e}')

    # Also try separate history JSON
    if history is None:
        history_path = os.path.join(RESULTS_DIR, f'{model_name}_history.json')
        if os.path.exists(history_path):
            with open(history_path, 'r') as f:
                history = json.load(f)

    if history is None:
        print(f'[MISSING] No training history for {model_name}')
        continue

    has_history = True
    color = MODEL_COLORS[model_name]
    display_name = DISPLAY_NAMES[model_name]
    epochs = range(1, len(history['val_acc']) + 1)

    # Validation accuracy
    axes[0].plot(epochs, history['val_acc'], '-o', markersize=3,
                 color=color, label=display_name, linewidth=2)

    # Validation loss
    axes[1].plot(epochs, history['val_loss'], '-o', markersize=3,
                 color=color, label=display_name, linewidth=2)

if has_history:
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Validation Accuracy')
    axes[0].set_title('Validation Accuracy Across Epochs')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Validation Loss')
    axes[1].set_title('Validation Loss Across Epochs')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.suptitle('Training Curves — All Models', fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'comparison_training_curves.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Training curves saved.')
else:
    plt.close(fig)
    print('No training history data available for any model.')
    print('Training curves will be available after models are trained with history saving enabled.')

In [ ]:
# ============================================================================
# Cell 10: Save comparison — CSV, summary PNG, final recommendation
# ============================================================================

# Save comparison table as CSV
csv_path = os.path.join(RESULTS_DIR, 'model_comparison.csv')
save_df = ranking.reset_index()
save_df.to_csv(csv_path, index=False)
print(f'Comparison table saved: {csv_path}')

# Generate summary figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

plot_order = ranking['Model'].tolist()
# Map display name back to model_key for colors
key_map = {v: k for k, v in DISPLAY_NAMES.items()}
plot_colors = [MODEL_COLORS[key_map[name]] for name in plot_order]

# Accuracy
acc_vals = [ranking.loc[ranking['Model'] == m, 'Accuracy'].values[0] for m in plot_order]
axes[0, 0].barh(plot_order, acc_vals, color=plot_colors, edgecolor='white')
axes[0, 0].set_title('Accuracy', fontweight='bold')
axes[0, 0].set_xlim(0, 1)
for i, val in enumerate(acc_vals):
    axes[0, 0].text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=10)

# F1 Macro
f1_vals = [ranking.loc[ranking['Model'] == m, 'F1 (Macro)'].values[0] for m in plot_order]
axes[0, 1].barh(plot_order, f1_vals, color=plot_colors, edgecolor='white')
axes[0, 1].set_title('F1 (Macro)', fontweight='bold')
axes[0, 1].set_xlim(0, 1)
for i, val in enumerate(f1_vals):
    axes[0, 1].text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=10)

# Inference time
inf_vals = [ranking.loc[ranking['Model'] == m, 'Inference (ms)'].values[0] for m in plot_order]
axes[1, 0].barh(plot_order, inf_vals, color=plot_colors, edgecolor='white')
axes[1, 0].set_title('Inference Time (ms)', fontweight='bold')
for i, val in enumerate(inf_vals):
    axes[1, 0].text(val + 0.2, i, f'{val:.2f}', va='center', fontsize=10)

# Weighted scores
ws_vals = [ranking.loc[ranking['Model'] == m, 'Weighted Score'].values[0] for m in plot_order]
bars = axes[1, 1].barh(plot_order, ws_vals, color=plot_colors, edgecolor='white')
axes[1, 1].set_title('Weighted Score', fontweight='bold')
axes[1, 1].set_xlim(0, 1)
for i, val in enumerate(ws_vals):
    axes[1, 1].text(val + 0.01, i, f'{val:.4f}', va='center', fontsize=10)
# Highlight winner
bars[0].set_edgecolor('black')
bars[0].set_linewidth(2)

plt.suptitle('Model Comparison Summary — Student Attention Detection',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
summary_path = os.path.join(RESULTS_DIR, 'model_comparison_summary.png')
plt.savefig(summary_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Summary figure saved: {summary_path}')

# Final recommendation
winner = ranking.iloc[0]
runner_up = ranking.iloc[1] if len(ranking) > 1 else None

print('\n' + '=' * 70)
print('FINAL RECOMMENDATION')
print('=' * 70)
print(f'\n  Best model: {winner["Model"]}')
print(f'  Weighted Score: {winner["Weighted Score"]:.4f}')
print(f'  Accuracy: {winner["Accuracy"]:.4f}')
print(f'  F1 (Macro): {winner["F1 (Macro)"]:.4f}')
print(f'  Inference: {winner["Inference (ms)"]:.2f} ms')
print(f'  Params: {winner["Params (M)"]:.2f}M')

if runner_up is not None:
    diff = winner['Weighted Score'] - runner_up['Weighted Score']
    print(f'\n  Runner-up: {runner_up["Model"]} (score: {runner_up["Weighted Score"]:.4f}, gap: {diff:.4f})')

print(f'\n  Use the {winner["Model"]} ONNX model for deployment.')
print('=' * 70)

# Sonraki Adimlar

## En Iyi Modeli Lokale Kopyalama

Karsilastirma sonucunda secilen en iyi modelin ONNX dosyasini lokale kopyalamak icin:

```bash
# Google Drive'dan ONNX modelini indir
# (Colab'da calistir veya Drive'dan manuel indir)

# Proje dizinine kopyala
cp /content/drive/MyDrive/Tubitak-2209B/onnx_models/<best_model>.onnx models/emotion_model.onnx
```

## Deploy Adimlari

1. **ONNX modelini indir** — Yukaridaki komutu kullanarak en iyi modeli `models/` dizinine kopyala
2. **Flask API'yi baslat** — `python -m src.api.run --model-path models/emotion_model.onnx`
3. **Dashboard'u test et** — Tarayicida `http://localhost:5000` adresini ac
4. **Yuz tanima entegrasyonu** — Ogrenci bazli takip icin InsightFace modulu ekle

## Yeniden Egitim

Farkli hiperparametreler veya veri seti ile tekrar egitim yapmak isterseniz `02_model_training.ipynb` notebook'unu kullanin. Egitim tamamlandiktan sonra bu karsilastirma notebook'unu yeniden calistirarak guncel sonuclari gorebilirsiniz.

## Notlar
- ONNX modeli CPU uzerinde calisir, GPU gerektirmez
- Inference suresi ONNX Runtime ile PyTorch'tan daha hizlidir
- Model 3 sinif tahmin eder: **negative**, **neutral**, **positive**
- Dikkat skoru icin `src/attention/` modulu kullanilir